# 04 — Short Squeeze Detection & Current Watchlist

Short squeezes occur when rising prices force short sellers to cover their positions, amplifying the move.  They tend to happen in stocks with:
* High short interest relative to float (large position to unwind)
* High days-to-cover (slow exit even without price impact)
* Recent upward price pressure (mark-to-market losses accumulating)
* Volume surges (covering activity visible in the tape)

This notebook trains a gradient-boosted classifier on historical squeeze events and evaluates it via walk-forward OOS cross-validation.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from securities_lending.models import SqueezeDetector, WalkForwardEvaluator
from securities_lending.models.squeeze_detector import SqueezeEventLabeler

plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.3})

features = pd.read_parquet("../data/processed/features.parquet")
features["date"] = pd.to_datetime(features["date"])
print(f"Features: {features.shape}")

## 1. Squeeze Event Labeling

A "squeeze event" is operationally defined as a 5-day forward return > 15% occurring simultaneously with DTC > 5 and volume > 2× 20-day average.  This combines the return signal with the microstructure conditions typical of short covering.

In [ ]:
labeler = SqueezeEventLabeler(
    return_threshold=0.15,
    dtc_threshold=5.0,
    volume_spike_threshold=2.0,
)
labels = labeler.label(features)
print(f"Squeeze events: {labels.sum()} / {len(labels)} ({labels.mean():.2%})")

# Show the distribution of squeeze events by ticker
features_labeled = features.copy()
features_labeled["squeeze"] = labels.values
squeeze_by_ticker = features_labeled.groupby("symbol")["squeeze"].sum().sort_values(ascending=False)
print("
Top 10 squeeze-prone tickers:")
print(squeeze_by_ticker.head(10))

In [ ]:
# Historical event timeline
fig, ax = plt.subplots(figsize=(13, 4))
squeeze_by_date = features_labeled.groupby("date")["squeeze"].sum()
squeeze_by_date.plot(ax=ax, color="#C53030", alpha=0.8, linewidth=0.8)
squeeze_by_date.rolling(21).mean().plot(ax=ax, color="#C53030", linewidth=2,
                                        label="21d rolling count")
ax.set_title("Squeeze Events per Day Across Universe")
ax.set_ylabel("# Stocks with Squeeze Event")
ax.legend(fontsize=9)
plt.show()

## 2. Walk-Forward OOS Evaluation

We use expanding-window walk-forward cross-validation with:
* Training window: 252 days
* Test window: 63 days  
* Step: 21 days (monthly retrain)

Primary metric is PR-AUC (precision-recall area under curve), which is more informative than ROC-AUC for rare events like squeezes.

In [ ]:
evaluator = WalkForwardEvaluator(
    train_window=252,
    test_window=63,
    step_size=21,
    seed=42,
)
wf_result = evaluator.run(features=features, labeler=labeler)
print("Walk-forward summary:", wf_result.summary())

# Plot per-window metrics
if wf_result.windows:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, series, label in zip(
        axes,
        [wf_result.roc_auc_series, wf_result.pr_auc_series, wf_result.precision_at_10_series],
        ["ROC-AUC", "PR-AUC", "Precision@10%"],
    ):
        ax.plot(series.values, "o-", color="#2B6CB0", linewidth=1.5, markersize=4)
        ax.axhline(series.mean(), color="#C53030", linestyle="--", linewidth=1.5,
                   label=f"Mean={series.mean():.3f}")
        # Baseline precision for random model
        baseline = labels.mean()
        ax.axhline(baseline, color="grey", linestyle=":", linewidth=1,
                   label=f"Base={baseline:.3f}")
        ax.set_title(label)
        ax.set_xlabel("OOS Window")
        ax.legend(fontsize=8)
    fig.suptitle("Walk-Forward OOS — Squeeze Detector", fontsize=12)
    plt.tight_layout()
    plt.show()

## 3. Final Model — Feature Importance (SHAP)

After walk-forward evaluation confirms OOS performance, we train a final model on all available data and inspect feature importance via SHAP values.

In [ ]:
final_model = SqueezeDetector(seed=42)
try:
    final_model.fit(features, labels)
    # SHAP analysis
    shap_values = final_model.explain(features.sample(min(500, len(features)), random_state=42))
    from securities_lending.viz import plot_squeeze_shap
    fig = plot_squeeze_shap(shap_values, final_model._feature_cols)
    plt.show()
except Exception as e:
    print(f"SHAP analysis failed: {e}")

## 4. Current Watchlist — Top Squeeze Candidates

Apply the trained model to the latest cross-section and identify stocks with the highest squeeze probability.

In [ ]:
if final_model._model is not None:
    try:
        scores = final_model.predict_proba(features)
        features["squeeze_prob"] = scores.values
        latest_date = features["date"].max()
        latest = features[features["date"] == latest_date].sort_values(
            "squeeze_prob", ascending=False
        ).head(15)
        cols = ["symbol", "squeeze_prob", "si_dtc", "svr_z20",
                "si_pct_float", "borrow_rate_bps", "short_pressure"]
        cols = [c for c in cols if c in latest.columns]
        print(f"
Top squeeze candidates as of {latest_date.date()}:")
        print(latest[cols].to_string(index=False))
    except Exception as e:
        print(f"Scoring failed: {e}")
else:
    print("Model not trained — run the full pipeline first")